In [1]:
import numpy as np
import pandas as pd
import balancepy as bp
import balancepy.anaropia as bpa
import balancepy.anaropia_project as bpap
from joblib import Parallel, delayed
from pathlib import Path
import sys; sys.path.insert(0, '..')
from src.config import paths

## Load metadata

In [4]:
# run syncronization with quality flag file
# bp.sync_quality_flags_from_metadata(paths=paths)

metadata = bpap.get_metadata(paths=paths)

## Configuration for ramp files

In [8]:
# preprocessing configuration
preproc_config = bpa.AnaropiaPreprocessingConfig()
preproc_config.end_time_seconds = 210
preproc_config.cut_to_cycles = True
preproc_config.cycle_length_samples = 360
preproc_config.cycle_start_samples = 878
print(preproc_config)

# stimulus-response class configuration
sr_config = bpa.AnaropiaSRDataConfig(
    stimulus_name='stim_pitch', 
    response_name=bpa.COM_LEGACY_AP, 
    frequency_selection='all'
    )
print(sr_config)

AnaropiaPreprocessingConfig(
  samplingrate_Hz: 90
  resample: True
  end_time_seconds: 210
  filter_type: None
  filter_order: 2
  filter_cutoff_Hz: 8
  cut_to_cycles: True
  cycle_start_samples: 878
  cycle_length_samples: 360
)
AnaropiaSRDataConfig(stimulus_name='stim_pitch', response_name=COMConfig(shoulder_pos_column='shld_zpos', hip_pos_column='hip_zpos', shoulder_height_column='shld_ypos', hip_height_column='hip_ypos', rotation=True), frequency_selection='all', name=None, column_scales=None)


## Print example quality-check plot for ramp stimulus trials

Shows raw data, resampled signal, cycle boundaries, and per-cycle spectra for a single subject and trial.

In [9]:
subject_id = 'KA07RI18'
filename = bpap.get_filename_from_metadata(paths, subject_id, 'c11: ramp 10', metadata=metadata)
body_height_m = metadata.loc[metadata['Subject ID'] == subject_id, 'body_height_m'].values[0]

# Single file visualization
fig, _, _ = bpa.plot_datacheck(filename, body_height_m, sr_config=sr_config, preproc_config=preproc_config, save=False)
fig.show()


In [ ]:
# Generate datacheck plots for all subjects and conditions
if True:  # set to False to skip
    jobs = []
    for subject_id, condition, filepath, body_height_m in bpap.batch_iterator(
        paths,
        conditions=[
            'c1: no stim', 'c2: ramp 0.01 A', 'c3: ramp 0.01 B', 'c4: ramp 0.05 A',
            'c5: ramp 0.05 B', 'c6: ramp 0.15', 'c7: ramp 0.25', 'c8: ramp 0.5',
            'c9: ramp 1', 'c10: ramp 5', 'c11: ramp 10'
        ],
        skip_flags=None,
    ):
        name = f"{subject_id}_{condition.replace(': ', '_').replace(' ', '_')}"
        if not (paths.datacheck_plots / f"datacheck_{name}.png").exists():
            jobs.append((filepath, body_height_m, name))

    _ = Parallel(n_jobs=3)(
        delayed(bpa.plot_datacheck)(
            filepath,
            body_height_m,
            sr_config=sr_config,
            preproc_config=preproc_config,
            name=name,
            output_dir=paths.datacheck_plots,
        )
        for filepath, body_height_m, name in jobs
    )

# Configuration of prts stimulus files

In [11]:
# preprocessing configuration
preproc_config = bpa.AnaropiaPreprocessingConfig()
preproc_config.end_time_seconds = 300
preproc_config.cut_to_cycles = True
preproc_config.cycle_length_samples = 1800
preproc_config.cycle_start_samples = 1800
print(preproc_config)

# stimulus-response class configuration
sr_config = bpa.AnaropiaSRDataConfig(
    stimulus_name='stim_pitch', 
    response_name=bpa.COM_LEGACY_AP, 
    frequency_selection='all'
    )
print(sr_config)

AnaropiaPreprocessingConfig(
  samplingrate_Hz: 90
  resample: True
  end_time_seconds: 300
  filter_type: None
  filter_order: 2
  filter_cutoff_Hz: 8
  cut_to_cycles: True
  cycle_start_samples: 1800
  cycle_length_samples: 1800
)
AnaropiaSRDataConfig(stimulus_name='stim_pitch', response_name=COMConfig(shoulder_pos_column='shld_zpos', hip_pos_column='hip_zpos', shoulder_height_column='shld_ypos', hip_height_column='hip_ypos', rotation=True), frequency_selection='all', name=None, column_scales=None)


## Print example quality-check plot for prts stimulus trials

Shows raw data, resampled signal, cycle boundaries, and per-cycle spectra for a single subject and trial.

In [12]:
subject_id = 'KA07RI18'
filename = bpap.get_filename_from_metadata(paths, subject_id, 'c12: prts', metadata=metadata)
body_height_m = metadata.loc[metadata['Subject ID'] == subject_id, 'body_height_m'].values[0]

# Single file visualization
fig, _, _ = bpa.plot_datacheck(filename, body_height_m, sr_config=sr_config, preproc_config=preproc_config, save=False)
fig.show()


In [13]:
# Generate datacheck plots for all subjects and conditions
if True:  # set to False to skip
    jobs = []
    for subject_id, condition, filepath, body_height_m in bpap.batch_iterator(
        paths,
        conditions=[
            'c12: prts'
        ],
        skip_flags=None,
    ):
        name = f"{subject_id}_{condition.replace(': ', '_').replace(' ', '_')}"
        if not (paths.datacheck_plots / f"datacheck_{name}.png").exists():
            jobs.append((filepath, body_height_m, name))

    _ = Parallel(n_jobs=3)(
        delayed(bpa.plot_datacheck)(
            filepath,
            body_height_m,
            sr_config=sr_config,
            preproc_config=preproc_config,
            name=name,
            output_dir=paths.datacheck_plots,
        )
        for filepath, body_height_m, name in jobs
    )